# 02. Reviewing the collected data

This notebook visualizes the data collected with `01_collect_data.ipynb` (`video.mp4` + `actions.csv` for each episode).
Before you go on to training (`03_train_bc.ipynb`), check here that you collected the data you wanted.
This notebook only reads data. It does not change data collection or training.
The last cell packs `data/` into `mario_data.zip`, the file you upload for training.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("../data")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
EPISODES_DIR = DATA_DIR / "episodes"

manifest = pd.read_csv(MANIFEST_PATH)
manifest

## Summary statistics

In [ ]:
print(f"episodes: {len(manifest)}")
print(f"clear rate (flag_get): {manifest['flag_get'].mean():.1%}")
manifest[["num_frames", "score", "x_pos_max", "coins", "time_remaining"]].describe()

## How good was the play? (Skill score)

This rates one episode with a score from 0 to 100.

```
progress = min(x_pos_max / 3161, 1)                          # progress rate (3161 = x position of the goal)
clear    = 1 if flag_get else 0                              # did Mario clear the stage?
t_used   = 400 - time_remaining                              # in-game time used
speed    = clear × clip((T_SLOW - t_used) / (T_SLOW - T_FAST), 0, 1)
points   = clear × min(score / SCORE_TARGET, 1)               # in-game score (coins, enemies, etc.)

Skill = 100 × (0.40·progress + 0.10·clear + 0.20·speed + 0.30·points)
```

- **Just clearing the stage gives 50 points** (progress 40% + clear 10%). Clearing 1-1 is not very
  hard, so the other 50 points come from speed (20%) and in-game score (30%).
  Reaching the goal gets its own bonus (10%) because "died just before the goal" and "cleared"
  are different in quality.
- **Speed and score only count for cleared episodes.** If Mario died, a short time only means he died early,
  not that the play was good, so we multiply by `clear` to turn it off. The score works the same way:
  an episode that does not clear is rated only on progress, so its maximum is 40 points.
  Time comes from the in-game timer, not from `num_frames`. (During area changes, such as going into a pipe,
  the environment moves frames forward internally, so the number of recorded frames does not match the real time.)
- **The in-game score (`score`) rates collecting coins and defeating enemies together.** A coin gives 200 points,
  a stomp gives 100 or more (it doubles for stomps in a row), a broken block gives 50, and a mushroom, flower, or star gives 1000.
  The episode ends the moment Mario touches the flagpole, so the score does not include
  the flagpole points or the time bonus. Speed is already measured by `speed`, so time is not counted twice.
- Rows with `truncated=True` (stopped with Esc) are left out. The player just stopped, so they say nothing about skill.

The constants and the formula are defined in `competition_config.py` (shared with 01 and 04, do not edit it).
`SCORE_TARGET` (5000 points) is about 10 coins, about 10 enemies, and 2 power-ups.

In [ ]:
# The formula and its constants live in competition_config.py (shared with 01/04)
from competition_config import add_skill_score

# Rows cut short with ESC only mean the player stopped, so leave them out
scored = add_skill_score(manifest[~manifest["truncated"]])

print(f"scored: {len(scored)} episodes (excluded: {int(manifest['truncated'].sum())})")
print(f"mean skill: {scored['skill'].mean():.1f}   best: {scored['skill'].max():.1f}")
print(f"clear rate: {scored['clear'].mean():.0%}")

scored.sort_values("skill", ascending=False)[
    ["episode_id", "x_pos_max", "flag_get", "t_used", "coins", "score",
     "progress", "speed", "points", "skill"]
].round(2)

In [ ]:
# Skill score distribution, and whether play improves as episodes accumulate
fig, (ax_hist, ax_time) = plt.subplots(1, 2, figsize=(12, 4))

ax_hist.hist(scored["skill"], bins=20, range=(0, 100), color="#4c78a8")
ax_hist.set_xlabel("Skill score")
ax_hist.set_ylabel("episodes")
ax_hist.set_title("Skill score distribution")

chronological = scored.sort_values("timestamp").reset_index(drop=True)
colors = ["#e45756" if c else "#4c78a8" for c in chronological["clear"]]
ax_time.scatter(chronological.index, chronological["skill"], c=colors)
ax_time.plot(
    chronological.index,
    chronological["skill"].rolling(5, min_periods=1).mean(),
    color="#666",
    label="moving average (5)",
)
ax_time.set_xlabel("episode (recording order)")
ax_time.set_ylabel("Skill score")
ax_time.set_ylim(0, 100)
ax_time.set_title("Skill score over time (red = cleared)")
ax_time.legend()

plt.tight_layout()
plt.show()

## Action distribution (all episodes)

To find class imbalance before training, this counts how many frames were recorded for each of the 7 actions.

In [ ]:
from collections import Counter

from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

ACTION_NAMES = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]

action_counts = Counter()
for ep_dir in sorted(EPISODES_DIR.glob("ep_*")):
    actions_csv = ep_dir / "actions.csv"
    if not actions_csv.exists():
        continue
    action_counts.update(pd.read_csv(actions_csv)["action"].tolist())

counts = [action_counts.get(i, 0) for i in range(len(SIMPLE_MOVEMENT))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ACTION_NAMES, counts)
ax.set_ylabel("frames")
ax.set_title("Action frequency")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Sample frames from an episode

Change `episode_id` to look at a different episode.

In [ ]:
import cv2
import numpy as np

episode_id = manifest["episode_id"].iloc[0]  # change this to the episode you want to inspect
ep_dir = EPISODES_DIR / episode_id
actions_df = pd.read_csv(ep_dir / "actions.csv")

cap = cv2.VideoCapture(str(ep_dir / "video.mp4"))
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"{episode_id}: video={num_frames} frames, actions.csv={len(actions_df)} frames")

n_samples = min(8, num_frames)
sample_idx = np.linspace(0, num_frames - 1, num=n_samples, dtype=int)

sample_frames = []
for idx in sample_idx:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame_bgr = cap.read()
    sample_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB) if ok else None)
cap.release()

fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3))
if n_samples == 1:
    axes = [axes]
for ax, idx, frame in zip(axes, sample_idx, sample_frames):
    action_idx = int(actions_df.loc[actions_df["frame_idx"] == idx, "action"].iloc[0])
    if frame is not None:
        ax.imshow(frame)
    ax.set_title(ACTION_NAMES[action_idx], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Action sequence plot

This shows which action was taken at which time in the chosen episode (for example, when Mario jumped).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(actions_df["frame_idx"], actions_df["action"])
ax.set_xlabel("frame index")
ax.set_ylabel("action")
ax.set_yticks(range(len(SIMPLE_MOVEMENT)))
ax.set_yticklabels(ACTION_NAMES)
ax.set_title(f"Action sequence - {episode_id}")
plt.tight_layout()
plt.show()

## Cleaning the data

If you find an episode that is clearly wrong, for example it has only a few frames or a recording mistake,
delete its `data/episodes/ep_YYYYMMDD_HHMMSS/` folder (`video.mp4` and `actions.csv`)
and delete the matching row in `data/manifest.csv` by hand.

`03_train_bc.ipynb` trains on every episode in `data/episodes/`. Which episodes to train on is up to your team.

Episode IDs are made from the recording time, so each one is unique. Deleting an episode never causes its ID to be reused.

## Making the zip for training

When you have finished checking and cleaning the data, run the cell below. It packs the `data` folder into
`mario_data.zip` in the project folder. Upload that file to your Google Drive and use it in `03_train_bc.ipynb`.

In [ ]:
import zipfile

ZIP_PATH = Path("../mario_data.zip")

files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file())
with zipfile.ZipFile(ZIP_PATH, "w") as zf:
    for path in files:
        compression = zipfile.ZIP_STORED if path.suffix == ".mp4" else zipfile.ZIP_DEFLATED
        zf.write(path, path.relative_to(DATA_DIR.parent), compress_type=compression)

n_episodes = len([p for p in EPISODES_DIR.glob("ep_*") if p.is_dir()])
print(f"saved: {ZIP_PATH.resolve()}")
print(f"{n_episodes} episodes, {len(files)} files, {ZIP_PATH.stat().st_size / 1024**2:.1f} MB")